In [5]:
import os
import shutil
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
import itertools
import shutil
import numpy as np
import scipy.stats as stats

# Importaciones específicas de la librería de series temporales
from neuralforecast import NeuralForecast
from neuralforecast.models import iTransformer
from neuralforecast.losses.pytorch import MSE, MAE

# Filtrar advertencias irrelevantes para mantener la salida limpia
warnings.filterwarnings('ignore')

# --- CONFIGURACIÓN DE HARDWARE (GPU vs CPU) ---
# Esta lógica asegura que 'DEVICES' sea siempre 1 (entero), 
# evitando el error "devices cannot be None" en CPU.

if torch.cuda.is_available():
    ACCELERATOR = "gpu"
    DEVICES = 1
    gpu_name = torch.cuda.get_device_name(0)
    print(f"🚀 HARDWARE DETECTADO: GPU ({gpu_name})")
else:
    ACCELERATOR = "cpu"
    DEVICES = 1
    print("⚠️ HARDWARE DETECTADO: CPU (El entrenamiento será más lento)")

# Limpieza preventiva de carpetas de logs antiguos
if os.path.exists("lightning_logs"):
    shutil.rmtree("lightning_logs")
if os.path.exists("checkpoints"):
    shutil.rmtree("checkpoints")

print("✅ Celda 1 ejecutada correctamente. Librerías listas.")

🚀 HARDWARE DETECTADO: GPU (NVIDIA GeForce GTX 1660 SUPER)
✅ Celda 1 ejecutada correctamente. Librerías listas.


In [6]:
# 1. Cargar datos
filename = "dataset_tfm_56_survivors.csv"

df = pd.read_csv(filename)
df['ds'] = pd.to_datetime(df['ds'])

print(f"📂 Datos cargados: {len(df)} filas. {df['unique_id'].nunique()} empresas.")

# 2. Filtro de Seguridad (Outliers)
# Eliminamos filas donde el retorno sea mayor al 50% en un día (ruido extremo)
# Esto protege al modelo de gradientes infinitos.
df = df[(df['y'] <= 0.5) & (df['y'] >= -0.5)].copy()

# 3. Normalización Z-Score (Manual)
# Calculamos media y desviación estándar para CADA empresa por separado
stats = df.groupby('unique_id')['y'].agg(['mean', 'std']).reset_index()
df = pd.merge(df, stats, on='unique_id', how='left')

# Aplicamos la fórmula: y_norm = (y - media) / (desviación + epsilon)
# El 1e-8 es para evitar divisiones por cero si una serie fuera plana.
df['y_norm'] = (df['y'] - df['mean']) / (df['std'] + 1e-8)

# 4. Preparar DataFrame Final
# Guardamos el valor original en 'y_raw' por si acaso, y usamos 'y_norm' para entrenar
df_final = df.copy()
df_final['y_raw'] = df_final['y']  
df_final['y'] = df_final['y_norm'] 

# Limpieza final: solo columnas necesarias, ordenado y sin nulos
df_final = df_final[['unique_id', 'ds', 'y', 'y_raw']].dropna()
df_final = df_final.sort_values(by=['unique_id', 'ds']).reset_index(drop=True)

print("-" * 30)
print("✅ DATOS NORMALIZADOS Y LISTOS.")
print(f"   Filas finales: {len(df_final)}")
print(f"   Rango de entrada a la red (y): {df_final['y'].min():.3f} a {df_final['y'].max():.3f}")
print("   (Lo ideal es que este rango esté cerca de entre -3 y 3)")

# Muestra las primeras filas para verificar
print(df_final.head())

📂 Datos cargados: 316000 filas. 56 empresas.
------------------------------
✅ DATOS NORMALIZADOS Y LISTOS.
   Filas finales: 315959
   Rango de entrada a la red (y): -21.900 a 21.193
   (Lo ideal es que este rango esté cerca de entre -3 y 3)
  unique_id         ds         y     y_raw
0       A3M 2004-01-02 -1.173126 -0.025802
1       A3M 2004-01-05  0.587628  0.013129
2       A3M 2004-01-06 -0.006163  0.000000
3       A3M 2004-01-07  1.030615  0.022924
4       A3M 2004-01-08 -0.198761 -0.004258


In [9]:
# 1. DEFINIMOS EL ESPACIO DE BÚSQUEDA
# Estos son los valores que vamos a combinar
param_grid = {
    # 'input_size': [30, 60, 90, 120],      # Corto, Medio, Largo plazo
    'input_size': [60],      # Corto, Medio, Largo plazo
    'hidden_size': [64, 128, 256, 512, 768],   # Ligero, Estándar, Pesado
    'e_layers': [1, 2, 3, 4, 5],           # Superficial vs Profundo
    'n_heads': [2, 4, 8, 16, 32],             # Poca vs Mucha granularidad
    'context': [3]
}

# 2. GENERAMOS LAS COMBINACIONES VÁLIDAS
# Usamos itertools para crear el producto cartesiano
keys, values = zip(*param_grid.items())
all_combinations = [dict(zip(keys, v)) for v in itertools.product(*values)]

# Filtramos las combinaciones matemáticas imposibles (hidden debe ser divisible por heads)
valid_configs = []
for cfg in all_combinations:
    if cfg['hidden_size'] % cfg['n_heads'] == 0:
        valid_configs.append(cfg)

print(f"🧪 SE HAN GENERADO {len(valid_configs)} COMBINACIONES VÁLIDAS.")
print("-" * 50)

🧪 SE HAN GENERADO 125 COMBINACIONES VÁLIDAS.
--------------------------------------------------


In [11]:

with open('logs.txt', 'a', encoding='utf8') as file:
    file.write(f"escenario;input_size;hidden_size;h_heads;e_layers;Error Medio de todo el mercado (MAE);Empresa más predecible (Error);Empresa más difícil (Error);Correlación\n")

for i, cfg in enumerate(valid_configs):
    print(f"[{i+1}/{len(valid_configs)}] Probando: {cfg} ...", end=" ")

    input_size = cfg['input_size']
    hidden_size = cfg['hidden_size']
    n_heads = cfg['n_heads']
    e_layers = cfg['e_layers']
    context = cfg['context']
    error_medio_de_todo_el_mercado = None
    predecible_mas = None
    predecible_meno = None
    correlacion = None

    CASE_SWITCH = context

    # Lógica del Split
    if CASE_SWITCH == 1:
        SCENARIO_NAME = "Caso1_Val_Crisis"
        VAL_YEAR = 2020

    elif CASE_SWITCH == 2:
        SCENARIO_NAME = "Caso2_Train_Crisis"
        VAL_YEAR = 2021

    elif CASE_SWITCH == 3:
        SCENARIO_NAME = "Caso3_Reciente"
        VAL_YEAR = 2023

    else:
        raise ValueError("❌ ERROR: CASE_SWITCH debe ser 1, 2 o 3.")

    # --- EJECUCIÓN DEL CORTE ---

    # 1. Definimos la fecha fin de los datos que verá el modelo (Train + Val)
    # El modelo NO verá nada posterior al 31 de diciembre del año de validación
    train_val_end_date = f"{VAL_YEAR}-12-31"

    # 2. Filtramos el DataFrame
    # df_input contiene TODO lo necesario para .fit() (Entrenamiento + Validación)
    df_input = df_final[df_final['ds'] <= train_val_end_date].copy()

    # 3. Calculamos el tamaño de validación (val_size)
    # Contamos cuántos días laborables únicos hay en el año de validación elegido
    val_start_date = f"{VAL_YEAR}-01-01"
    validation_data = df_input[df_input['ds'] >= val_start_date]
    val_size = len(validation_data['ds'].unique())

    HORIZONTE = 5       # Predicción a 1 semana (5 días laborables)
    VENTANA_HIST = 60   # Miramos 3 meses atrás (60 días laborables)

    # Definición del iTransformer
    model = iTransformer(
        # 1. Dimensiones
        h = HORIZONTE,
        input_size = input_size,
        n_series = 56,            # Nº exacto de tus empresas
        
        # 2. Hiperparámetros de la Arquitectura (Transformer puro)
        hidden_size = hidden_size,        # Tamaño del vector latente
        n_heads = n_heads,              # Cabezales de atención
        e_layers = e_layers,             # Capas de codificación
        
        # 3. Funciones de Pérdida
        loss = MSE(),             # Optimizamos reduciendo el Error Cuadrático
        valid_loss = MAE(),       # Validamos mirando el Error Absoluto (más humano)
        
        # 4. Configuración de Entrenamiento
        max_steps = 500,          # Máximo de iteraciones
        early_stop_patience_steps = 10, # Parar si no mejora en 10 validaciones
        batch_size = 32,
        
        # 5. AJUSTES CRÍTICOS DE ESTABILIDAD
        learning_rate = 1e-4,     # 0.0001 (Lento y seguro)
        gradient_clip_val = 1.0,  # Evita picos de error
        scaler_type = 'identity', # <--- IMPORTANTE: No re-escalar (ya lo hicimos nosotros)
        
        # --- LA CORRECCIÓN CLAVE ---
        log_every_n_steps = 1,  # <--- ESTO OBLIGA A GUARDAR EL LOG SIEMPRE

        # 6. Hardware
        random_seed = 42,
        accelerator = ACCELERATOR,
        devices = DEVICES
    )

    # Inicialización del Motor (Trainer)
    nf = NeuralForecast(
        models=[model], 
        freq='B' # Business Days (Días laborables)
    )

    # Limpieza preventiva
    if os.path.exists("lightning_logs"):
        shutil.rmtree("lightning_logs")

    nf.fit(df=df_input, val_size=val_size)

    # 2. Guardar
    save_path = f"checkpoints/{SCENARIO_NAME}"
    nf.save(path=save_path, model_index=None, overwrite=True)

    # 1. Cargar el modelo que acabamos de guardar
    nf_loaded = NeuralForecast.load(path=f"checkpoints/{SCENARIO_NAME}")

    # 2. Preparar el escenario de prueba
    # Queremos ver qué tal predice el INICIO del periodo de validación.
    # Cortamos los datos justo antes del 1 de Enero del año de validación.
    cutoff_date = f"{VAL_YEAR}-01-01"
    df_history = df_input[df_input['ds'] < cutoff_date].copy()


    # 3. Predecir (Usando la herramienta nativa de la librería)
    # El modelo leerá la historia y predecirá los siguientes 5 días (HORIZONTE)
    forecasts = nf_loaded.predict(df=df_history)
    forecasts = forecasts.reset_index()

    # 4. Cruzar con la realidad
    # Traemos los datos reales de df_input para esos días
    comparison = pd.merge(
        forecasts, 
        df_input[['unique_id', 'ds', 'y']], 
        on=['unique_id', 'ds'], 
        how='inner'
    )

    if comparison.empty:
        print("❌ ERROR: Las fechas de predicción no coinciden con los datos reales.")
        with open('logs.txt', 'a', encoding='utf8') as file:
            file.write(f"ERROR: Las fechas de predicción no coinciden con los datos reales.\n")
    else:
        # 5. Calcular Métricas Reales
        mse = ((comparison['y'] - comparison['iTransformer'])**2).mean()
        mae = (comparison['y'] - comparison['iTransformer']).abs().mean()
        
        # 6. Gráfica Visual
        # Cogemos una empresa al azar para no saturar el gráfico
        sample_id = comparison['unique_id'].unique()[0]
        sub_real = df_input[(df_input['unique_id'] == sample_id) & 
                            (df_input['ds'] >= pd.to_datetime(cutoff_date) - pd.Timedelta(days=15)) &
                            (df_input['ds'] <= pd.to_datetime(cutoff_date) + pd.Timedelta(days=10))]
        
        sub_pred = comparison[comparison['unique_id'] == sample_id]

        # --- EVALUACIÓN MASIVA DE LAS 56 EMPRESAS ---

        # 1. Preparar datos para predecir
        # Cortamos la historia justo antes del año de validación
        cutoff_date = f"{VAL_YEAR}-01-01"
        df_history = df_input[df_input['ds'] < cutoff_date].copy()

        # 2. Predecir para TODAS las empresas a la vez
        # El modelo devuelve un DataFrame con 56 predicciones (una por unique_id)
        forecasts = nf_loaded.predict(df=df_history)
        forecasts = forecasts.reset_index()

        # 3. Cruzar con la realidad
        results_all = pd.merge(
            forecasts, 
            df_input[['unique_id', 'ds', 'y']], 
            on=['unique_id', 'ds'], 
            how='inner'
        )

        if results_all.empty:
            print("❌ ERROR: No hay coincidencias de fechas.")
            with open('logs.txt', 'a', encoding='utf8') as file:
                file.write(f"ERROR: No hay coincidencias de fechas.\n")
        else:
            # 4. Calcular error por empresa (Agrupamos por ID)
            # Calculamos el MAE (Error Absoluto Medio) para cada acción
            errors_per_id = results_all.groupby('unique_id').apply(
                lambda x: (x['y'] - x['iTransformer']).abs().mean()
            ).reset_index()
            errors_per_id.columns = ['unique_id', 'MAE']
            
            # Ordenamos: Arriba las que tienen MENOS error (Mejores)
            errors_per_id = errors_per_id.sort_values('MAE')

            error_medio_de_todo_el_mercado = errors_per_id['MAE'].mean()
            predecible_mas = errors_per_id.iloc[0]['MAE']
            predecible_meno = errors_per_id.iloc[-1]['MAE']

            # 1. Calcular la Volatilidad de la entrada (Lo que el modelo "ve" antes de predecir)
            # Usamos df_history (los datos previos al corte de validación)
            volatility_metrics = []

            cutoff_date = f"{VAL_YEAR}-01-01"
            df_history = df_input[df_input['ds'] < cutoff_date].copy()

            for uid in df_history['unique_id'].unique():
                # Cogemos los últimos 60 días de esa empresa
                company_data = df_history[df_history['unique_id'] == uid].tail(VENTANA_HIST)
                
                # Calculamos desviación estándar (Volatilidad)
                vol = company_data['y'].std()
                volatility_metrics.append({'unique_id': uid, 'Input_Volatility': vol})

            df_vol = pd.DataFrame(volatility_metrics)

            # 2. Calcular el Error Real (MAE) de la predicción (Lo que ya calculamos antes)
            # Reutilizamos 'results_all' de la celda anterior o lo recalculamos si hace falta
            # (Asumo que results_all existe de la Celda 7. Si no, avísame)
            if 'results_all' in locals():
                errors_per_id = results_all.groupby('unique_id').apply(
                    lambda x: (x['y'] - x['iTransformer']).abs().mean()
                ).reset_index()
                errors_per_id.columns = ['unique_id', 'Prediction_Error_MAE']

                # 3. Unir Volatilidad (Pasado) vs Error (Futuro)
                analysis_df = pd.merge(df_vol, errors_per_id, on='unique_id')

                # 4. Análisis de Correlación
                correlation = analysis_df['Input_Volatility'].corr(analysis_df['Prediction_Error_MAE'])

                correlacion = correlation

                with open('logs.txt', 'a', encoding='utf8') as file:
                    file.write(f"{context};{input_size};{hidden_size};{n_heads};{e_layers};{error_medio_de_todo_el_mercado:.5f};{predecible_mas:.5f};{predecible_meno:.5f};{correlation:.4f}\n")
            else:
                print("❌ Por favor, ejecuta primero la Celda 7 para tener 'results_all'.")
                file.write(f"Por favor, ejecuta primero la Celda 7 para tener 'results_all'.\n")
                        

Seed set to 42
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name          | Type                   | Params | Mode  | FLOPs
-------------------------------------------------------------------------
0 | loss          | MSE                    | 0      | train | 0    
1 | valid_loss    | MAE                    | 0      | train | 0    
2 | padder_train  | ConstantPad1d          | 0      | train | 0    
3 | scaler        | TemporalNorm           | 0      | train | 0    
4 | enc_embedding | DataEmbedding_inverted | 3.9 K  | train | 0    
5 | encoder       | TransEncoder           | 281 K  | train | 0    
6 | projector     | Linear                 | 325    | train | 0    
-------------------------------------------------------------------------
285 K     Trainable params
0         Non-trainable params
285 K     Total params
1.142     Total estimated model params size (MB)
24        Modules in train mode
0       

Epoch 499: 100%|██████████| 1/1 [00:00<00:00, 18.03it/s, v_num=0, train_loss_step=1.090, train_loss_epoch=1.090, valid_loss=0.542]

`Trainer.fit` stopped: `max_steps=500` reached.


Epoch 499: 100%|██████████| 1/1 [00:00<00:00, 17.39it/s, v_num=0, train_loss_step=1.090, train_loss_epoch=1.090, valid_loss=0.542]


Seed set to 42
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


...

Seed set to 42
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Predicting DataLoader 0: 100%|██████████| 1/1 [00:00<00:00, 117.86it/s]

Trainer already configured with model summary callbacks: [<class 'pytorch_lightning.callbacks.model_summary.ModelSummary'>]. Skipping setting a default `ModelSummary` callback.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Predicting DataLoader 0: 100%|██████████| 1/1 [00:00<00:00, 125.32it/s]
